# Redrob's Hackathon : Data & AI Challenge Code
Targeting AI Engineers with 'Shipper' attitude. Includes ghost-worker detection, prestige tiering, and notice-buyout heuristics.

In [ ]:
import os
import gzip
import json
import pandas as pd
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. SETTINGS & PATHS
RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'candidates.jsonl')
IF_GZ_PATH = os.path.join('..', 'data', 'raw', 'candidates.jsonl.gz')
CANDIDATES_PATH = RAW_DATA_PATH if os.path.exists(RAW_DATA_PATH) else IF_GZ_PATH
MODEL_PATH = os.path.join('..', 'models', 'all-MiniLM-L6-v2')
OUTPUT_CSV_PATH = '../data/final/submission3.csv'

SERVICE_COMPANIES = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini', 'hcl', 'tech mahindra', 'mindtree'}

# 2. LOAD MODEL (OFFLINE MODE)
print("Loading Transformer Model...")
if os.path.exists(MODEL_PATH):
    model = SentenceTransformer(MODEL_PATH)
else:
    print("WARNING: Local model not found. Downloading via internet (valid for local dev only)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. STAGE 1: HEURISTIC FILTERING (100,000 -> 2,000)
print("Stage 1: Heuristic Filtering...")
processed = []
open_func = gzip.open if CANDIDATES_PATH.endswith('.gz') else open

with open_func(CANDIDATES_PATH, 'rt', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        try:
            c = json.loads(line)
            profile = c.get('profile', {})
            yoe = float(profile.get('years_of_experience', 0) or 0)
            
            # Hard filters for performance
            if yoe < 4 or yoe > 20: continue
            
            signals = c.get('redrob_signals', {}) or {}
            notice = signals.get('notice_period_days', 90)
            is_service = any(sc in str(profile.get('current_company','')).lower() for sc in SERVICE_COMPANIES)
            
            # Fast heuristic score
            h_score = (1.2 if not is_service else 0.8) * (1.15 if notice <= 30 else 1.0)
            h_score *= float(signals.get('recruiter_response_rate', 0.5))

            processed.append({
                'candidate_id': c['candidate_id'],
                'yoe': yoe,
                'text': f"{profile.get('headline','')} {profile.get('summary','')} {profile.get('current_title','')}".lower(),
                'h_score': h_score,
                'is_product': not is_service,
                'notice': notice,
                'skills': [s.get('name','') for s in c.get('skills', [])][:3]
            })
        except: continue

df = pd.DataFrame(processed)
print(f"Pruned to {len(df)} candidates.")

# Fast TF-IDF pass to get the top 2000 for the Transformer
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(["Senior AI Engineer ranking retrieval search"] + df['text'].tolist())
df['tfidf_sim'] = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])[0]
df['stage1_score'] = df['tfidf_sim'] * df['h_score']

df_rerank = df.sort_values('stage1_score', ascending=False).head(2000).copy()
print("Stage 1 Complete. Reranking top 2,000 using Transformer...")

# 4. STAGE 2: TRANSFORMER RERANKING (2,000 -> 100)
jd_text = "Senior AI Engineer. Expertise in ranking systems, vector retrieval, embeddings, RAG, and LLM fine-tuning. Production experience required."
jd_emb = model.encode(jd_text, convert_to_tensor=True)
candidate_embs = model.encode(df_rerank['text'].tolist(), convert_to_tensor=True, show_progress_bar=True)

df_rerank['semantic_score'] = util.cos_sim(jd_emb, candidate_embs)[0].cpu().numpy()

# 5. FINAL SCORE & REASONING
df_rerank['final_score'] = df_rerank['semantic_score'] * df_rerank['h_score']

def generate_v3_reasoning(row):
    return f"Senior AI Engineer ({row['yoe']}y) with strong semantic match. {row['is_product'] and 'Product-company background' or ''}. Expert in {', '.join(row['skills'])}."

df_final = df_rerank.sort_values('final_score', ascending=False).head(100).copy()
df_final['rank'] = range(1, 101)
df_final['reasoning'] = df_final.apply(generate_v3_reasoning, axis=1)

df_final[['candidate_id', 'rank', 'final_score', 'reasoning']].rename(columns={'final_score':'score'}).to_csv(OUTPUT_CSV_PATH, index=False)
print(f"SUCCESS: Transformer Ranking Saved to {OUTPUT_CSV_PATH}")


Initiating Final Tier Ranker...
Scanned 0 profiles...
Scanned 25000 profiles...
Scanned 50000 profiles...
Scanned 75000 profiles...
Final Rerank Pool: 74955
SUCCESS: Ultimate Ranking Saved to ../data/final/submission.csv
